In [1]:
pip install rasterio numpy scikit-image

In [2]:
import os
import warnings

import numpy as np
import rasterio
from rasterio.windows import Window, bounds, from_bounds
from skimage.exposure import match_histograms

In [16]:
# ============================================================
# 1. INPUT FILE PATHS
# ============================================================

# Single HR raster containing all four bands
hr_path = r"/content/NAIP UTM.tif"

# Four individual LR raster files (one band per file)
lr_paths = [
    r"/content/B2.tif",
    r"/content/B3.tif",
    r"/content/B4.tif",
    r"/content/B5.tif"
]

output_path = r"/content/NAIP_matched.tif"

In [17]:
# ------------------------------------------------
# PATCH CONFIGURATION
# ------------------------------------------------

N_COLS = 6
N_ROWS = 8

TOTAL_PATCHES = N_COLS * N_ROWS

MIN_VALID_PIXELS = 10

In [18]:
with rasterio.open(hr_path) as hr:

    print("HIGH RESOLUTION RASTER")
    print("----------------------")
    print("Dimensions:", hr.width, "x", hr.height)
    print("Bands:", hr.count)
    print("Data type:", hr.dtypes)
    print("Resolution:", hr.res)
    print("CRS:", hr.crs)
    print("NoData:", hr.nodata)
    print("File size:",
          round(os.path.getsize(hr_path) / (1024**2), 2),
          "MB")

print("\nLOW RESOLUTION RASTERS")
print("----------------------")

for i, path in enumerate(lr_paths):

    with rasterio.open(path) as lr:

        print(f"\nBand {i+1}")
        print("Dimensions:", lr.width, "x", lr.height)
        print("Data type:", lr.dtypes)
        print("Resolution:", lr.res)
        print("CRS:", lr.crs)
        print("NoData:", lr.nodata)

HIGH RESOLUTION RASTER
----------------------
Dimensions: 9580 x 12320
Bands: 4
Data type: ('uint8', 'uint8', 'uint8', 'uint8')
Resolution: (0.5999998480193509, 0.5999998480193509)
CRS: EPSG:32612
NoData: None
File size: 450.3 MB

LOW RESOLUTION RASTERS
----------------------

Band 1
Dimensions: 192 x 248
Data type: ('float32',)
Resolution: (30.0, 30.0)
CRS: EPSG:32612
NoData: -3.4028234663852886e+38

Band 2
Dimensions: 192 x 248
Data type: ('float32',)
Resolution: (30.0, 30.0)
CRS: EPSG:32612
NoData: -3.4028234663852886e+38

Band 3
Dimensions: 192 x 248
Data type: ('float32',)
Resolution: (30.0, 30.0)
CRS: EPSG:32612
NoData: -3.4028234663852886e+38

Band 4
Dimensions: 192 x 248
Data type: ('float32',)
Resolution: (30.0, 30.0)
CRS: EPSG:32612
NoData: -3.4028234663852886e+38


In [19]:

def extract_lr_patch(lr_dataset, patch_bounds):

    left, bottom, right, top = patch_bounds

    window = from_bounds(
        left, bottom, right, top,
        transform=lr_dataset.transform
    )

    window = window.round_offsets().round_lengths()

    full_window = Window(
        0, 0,
        lr_dataset.width,
        lr_dataset.height
    )

    try:
        window = window.intersection(full_window)
    except Exception:
        return np.array([], dtype=np.float32)

    if window.width <= 0 or window.height <= 0:
        return np.array([], dtype=np.float32)

    # Read original LR values and validity mask
    lr_array = lr_dataset.read(
        1,
        window=window,
        masked=True
    )

    lr_data = lr_array.data.astype(np.float32)

    valid = ~np.ma.getmaskarray(lr_array)
    valid &= np.isfinite(lr_data)

    # Calculate LR pixel center coordinates
    transform = lr_dataset.window_transform(window)

    rows, cols = np.indices(lr_data.shape)

    x_coords = (
        transform.c +
        (cols + 0.5) * transform.a
    )

    y_coords = (
        transform.f +
        (rows + 0.5) * transform.e
    )

    # Geographic inclusion test
    inside_patch = (
        (x_coords >= left) &
        (x_coords < right) &
        (y_coords > bottom) &
        (y_coords <= top)
    )

    valid &= inside_patch

    return lr_data[valid]


In [23]:

def histogram_match_rasters(
    hr_path,
    lr_paths,
    output_path,
    n_cols=6,
    n_rows=8,
    min_valid_pixels=10
):

    with rasterio.open(hr_path) as hr:

        # -----------------------------------------
        # Validate HR raster
        # -----------------------------------------

        if hr.count != 4:
            raise ValueError("HR raster must contain 4 bands.")

        if hr.crs.to_epsg() != 32612:
            raise ValueError("HR raster must use EPSG:32612.")

        print("Original HR data type:", hr.dtypes)
        print("Output data type: float32")

        # -----------------------------------------
        # Configure output raster
        # -----------------------------------------

        profile = hr.profile.copy()

        profile.update(
            count=4,
            dtype="float32",
            compress="deflate",
            predictor=3
        )

        # Preserve NoData value, represented as float32
        if hr.nodata is not None:
            profile.update(nodata=float(hr.nodata))

        # -----------------------------------------
        # Open LR datasets
        # -----------------------------------------

        lr_datasets = []

        try:

            for i, path in enumerate(lr_paths):

                lr = rasterio.open(path)

                if lr.count != 1:
                    raise ValueError(
                        f"LR Band {i+1} must contain one band."
                    )

                if lr.crs != hr.crs:
                    raise ValueError(
                        f"CRS mismatch in LR Band {i+1}."
                    )

                lr_datasets.append(lr)

            # -------------------------------------
            # Calculate patch boundaries
            # -------------------------------------

            col_edges = np.linspace(
                0, hr.width, n_cols + 1
            ).round().astype(int)

            row_edges = np.linspace(
                0, hr.height, n_rows + 1
            ).round().astype(int)

            total_patches = n_cols * n_rows

            print(f"\nProcessing {total_patches} patches...")

            # -------------------------------------
            # Create output raster
            # -------------------------------------

            with rasterio.open(
                output_path,
                "w",
                **profile
            ) as dst:

                patch_number = 0

                for row in range(n_rows):

                    for col in range(n_cols):

                        patch_number += 1

                        r0 = row_edges[row]
                        r1 = row_edges[row + 1]

                        c0 = col_edges[col]
                        c1 = col_edges[col + 1]

                        hr_window = Window(
                            c0,
                            r0,
                            c1 - c0,
                            r1 - r0
                        )

                        patch_bounds = bounds(
                            hr_window,
                            hr.transform
                        )

                        print(
                            f"\nPatch {patch_number}/{total_patches}"
                        )

                        # =============================
                        # PROCESS FOUR BANDS
                        # =============================

                        for band in range(1, 5):

                            # Read HR patch and convert to float32
                            hr_patch = hr.read(
                                band,
                                window=hr_window
                            ).astype(np.float32)

                            # Preserve original validity mask
                            hr_mask = hr.read_masks(
                                band,
                                window=hr_window
                            ) > 0

                            hr_mask &= np.isfinite(hr_patch)

                            # Extract corresponding LR pixels
                            lr_reference = extract_lr_patch(
                                lr_datasets[band - 1],
                                patch_bounds
                            )

                            # Check valid pixel counts
                            if (
                                hr_mask.sum() < min_valid_pixels or
                                len(lr_reference) < min_valid_pixels
                            ):

                                print(
                                    f"  Band {band}: insufficient "
                                    "valid pixels; original retained."
                                )

                                dst.write(
                                    hr_patch,
                                    band,
                                    window=hr_window
                                )

                                continue

                            # Convert LR reference to float32
                            lr_reference = lr_reference.astype(
                                np.float32
                            )

                            # Extract valid HR values
                            hr_values = hr_patch[hr_mask]

                            # Perform histogram matching
                            matched_values = match_histograms(
                                hr_values,
                                lr_reference,
                                channel_axis=None
                            )

                            # Retain floating-point precision
                            hr_patch[hr_mask] = matched_values.astype(
                                np.float32
                            )

                            # Write matched patch
                            dst.write(
                                hr_patch,
                                band,
                                window=hr_window
                            )

                            print(
                                f"  Band {band}: "
                                f"HR valid = {hr_mask.sum():,}; "
                                f"LR valid = {len(lr_reference):,}"
                            )

                # Preserve original HR dataset validity mask
                dst.write_mask(hr.dataset_mask())

        finally:

            for lr in lr_datasets:
                lr.close()

    print("\n===================================")
    print("HISTOGRAM MATCHING COMPLETE")
    print("===================================")
    print("Output:", output_path)
    print("Output data type: float32")

In [24]:
# Run processing
histogram_match_rasters(
    hr_path=hr_path,
    lr_paths=lr_paths,
    output_path=output_path,
    n_cols=N_COLS,
    n_rows=N_ROWS,
    min_valid_pixels=MIN_VALID_PIXELS
)

Original HR data type: ('uint8', 'uint8', 'uint8', 'uint8')
Output data type: float32

Processing 48 patches...

Patch 1/48
  Band 1: HR valid = 2,459,380; LR valid = 960
  Band 2: HR valid = 2,459,380; LR valid = 960
  Band 3: HR valid = 2,459,380; LR valid = 960
  Band 4: HR valid = 2,459,380; LR valid = 960

Patch 2/48
  Band 1: HR valid = 2,457,840; LR valid = 960
  Band 2: HR valid = 2,457,840; LR valid = 960
  Band 3: HR valid = 2,457,840; LR valid = 960
  Band 4: HR valid = 2,457,840; LR valid = 960

Patch 3/48
  Band 1: HR valid = 2,459,380; LR valid = 960
  Band 2: HR valid = 2,459,380; LR valid = 960
  Band 3: HR valid = 2,459,380; LR valid = 960
  Band 4: HR valid = 2,459,380; LR valid = 960

Patch 4/48
  Band 1: HR valid = 2,459,380; LR valid = 960
  Band 2: HR valid = 2,459,380; LR valid = 960
  Band 3: HR valid = 2,459,380; LR valid = 960
  Band 4: HR valid = 2,459,380; LR valid = 960

Patch 5/48
  Band 1: HR valid = 2,457,840; LR valid = 960
  Band 2: HR valid = 2,457,84

In [25]:
with rasterio.open(hr_path) as original, \
     rasterio.open(output_path) as matched:

    print("Original data type:", original.dtypes)
    print("Output data type:", matched.dtypes)

    print("Dimensions identical:",
          original.shape == matched.shape)

    print("Transform identical:",
          original.transform == matched.transform)

    print("CRS identical:",
          original.crs == matched.crs)

    print("Resolution identical:",
          original.res == matched.res)

    print("Output bands:", matched.count)

    print("\nOutput file size:",
          round(os.path.getsize(output_path) / (1024**2), 2),
          "MB")

Original data type: ('uint8', 'uint8', 'uint8', 'uint8')
Output data type: ('float32', 'float32', 'float32', 'float32')
Dimensions identical: True
Transform identical: True
CRS identical: True
Resolution identical: True
Output bands: 4

Output file size: 1157.41 MB


In [26]:
import os

# Specify your target directory (e.g., 'sample_data' or your mounted Google Drive path)
directory = '/content'

print(f"{'File Name':<35} | {'Size (KB)':<12} | {'Full Path'}")
print("-" * 75)

for filename in os.listdir(directory):
  filepath = os.path.join(directory, filename)
  if os.path.isfile(filepath):
    size_bytes = os.path.getsize(filepath)
    size_kb = round(size_bytes / 1024, 2)
    print(f"{filename:<35} | {size_kb:<12} | {filepath}")

File Name                           | Size (KB)    | Full Path
---------------------------------------------------------------------------
NAIP_matched.tif                    | 1185191.71   | /content/NAIP_matched.tif
B2.tif                              | 187.49       | /content/B2.tif
B4.tif                              | 187.49       | /content/B4.tif
NAIP UTM.tif                        | 461110.17    | /content/NAIP UTM.tif
B5.tif                              | 187.49       | /content/B5.tif
B3.tif                              | 187.49       | /content/B3.tif
